In [1]:
!pip -q install kaggle nltk gensim pyLDAvis wordcloud

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 44.6 MB/s eta 0:00:00


In [2]:
!ls -lah

total 16K
drwxr-xr-x 1 root root 4.0K May  6 13:29 .
drwxr-xr-x 1 root root 4.0K May  9 07:56 ..
drwxr-xr-x 4 root root 4.0K May  6 13:29 .config
drwxr-xr-x 1 root root 4.0K May  6 13:29 sample_data


In [ ]:
from google.colab import files
files.upload()

In [ ]:
!ls

In [ ]:
!unzip "twitter-airline-sentiment (1).zip" -d /content/airline_data

In [ ]:
import pandas as pd
import glob, os
csv_files = glob.glob("/content/airline_data/*.csv")
print("CSV files found:", csv_files)
csv_path = csv_files[0]
df = pd.read_csv(csv_path)
print("Shape:", df.shape)
df.head()

In [ ]:
print("Columns:")
print(df.columns.tolist())
print("\nSentiment counts:")
print(df["airline_sentiment"].value_counts())
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
df = df[["text", "airline_sentiment"]].dropna().copy()
df.head()

In [ ]:
import re
import nltk
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and
    len(w) > 2]
    return " ".join(tokens)
df["clean_text"] = df["text"].apply(clean_text)
df[["text","clean_text"]].head()

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
all_words = " ".join(df["clean_text"])
wordcloud = WordCloud(width=1000, height=500,
background_color="white").generate(all_words)
plt.figure(figsize=(12,6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Airline Tweets")
plt.show()

In [ ]:
from gensim import corpora
from gensim.models import LdaModel
texts = [text.split() for text in df["clean_text"] if text.strip() != ""]
dictionary = corpora.Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]
lda_model = LdaModel(
corpus=corpus,
id2word=dictionary,
num_topics=5,
random_state=42,
passes=10
)
topics = lda_model.print_topics(num_words=8)
for t in topics:
    print(t)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
X = df["clean_text"]
y = df["airline_sentiment"]
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y
)
tfidf = TfidfVectorizer(max_features=3000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train_tfidf, y_train)


In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
accuracy_score)
import seaborn as sns
y_pred = clf.predict(X_test_tfidf)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

y_pred = clf.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)

# --- 3D Plot ---
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

_x = np.arange(cm.shape[0])
_y = np.arange(cm.shape[1])
_xx, _yy = np.meshgrid(_x, _y)
x, y = _xx.ravel(), _yy.ravel()

top = cm.ravel()
bottom = np.zeros_like(top)
width = depth = 0.6

colors = plt.cm.viridis(top / max(top))  # colorful gradient

ax.bar3d(x, y, bottom, width, depth, top, shade=True, color=colors)

ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_zlabel('Count')

ax.set_xticks(np.arange(len(clf.classes_)))
ax.set_yticks(np.arange(len(clf.classes_)))

ax.set_xticklabels(clf.classes_)
ax.set_yticklabels(clf.classes_)

plt.title("3D Confusion Matrix")
plt.show()

In [ ]:
df["airline_sentiment"].value_counts().plot(kind="bar", title="Sentiment Class Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()